In [1]:
import IPython.display as ipd

import os, sys
current_path = os.getcwd()
path_parts = current_path.split(os.sep)
idx = path_parts.index("occupancy-estimation")
homepath = os.sep.join(path_parts[:idx + 1]) +"/"
sys.path.append(homepath)

%load_ext autoreload 
%autoreload 2
%matplotlib inline


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from meteor import rsmap
from meteor.utils import cut_resolution
from systematic_plots import setup_logger

from meteor.diffmaps import compute_difference_map, max_negentropy_kweighted_difference_map
from meteor.tv import tv_denoise_difference_map
from meteor.validate import map_negentropy

In [3]:
# from photolyase import *
from photolyase import get_exp_structure_factors, calculate_dog_filter_mask_v2, ccp4_to_mask
from compare_conds import get_intersect_and_angle
from photolyase import get_hists, plot_hists, find_wasserstein_dip
from plotting3d import slice_3d, add_fit
from meteorize import *
# from output_eval import panels_3d
from systematic_plots import setup_logger


In [4]:
logger = setup_logger(logging.INFO)


In [5]:
folderloc =  homepath+'../photolyase_share_reviewers/'
dataloc_dark  = folderloc + '1_superdark/superdark_deposit.mtz'
pdbloc_dark  = folderloc + '1_superdark/superdark_deposit.pdb'
changing_bit = "6_10ns/10ns"
changing_bit = "2_3ps/3ps"
changing_bit = "3_300ps/300ps"
changing_bit = "7_30ns/30ns"
dataloc_light = folderloc + changing_bit + "_deposit.mtz"
pdbloc_light  = folderloc + changing_bit + '_deposit.pdb'


mask_loc_sphere = homepath+"../photolyase_masks/photolyase_TT_chainA_spherical_mask.ccp4"
mask_loc_handcraft = homepath+"../photolyase_masks/photolyase_chainA_spherical_masks.ccp4"
hs_limit = 2.4 
map_sampling = 3




In [6]:
ds_light = rs.read_mtz(dataloc_light)
ds_dark = rs.read_mtz(dataloc_dark)

ds_dark = cut_resolution(ds_dark,high_resolution_limit=hs_limit)
ds_light = cut_resolution(ds_light,high_resolution_limit=hs_limit)

ds_light

IMEAN       SIGIMEAN           F       SIGF       KFEXTR  \
H  K L                                                                   
0  0 2          NaN            NaN         NaN        NaN          NaN   
     4          NaN            NaN         NaN        NaN          NaN   
     6   563041.625  124030.234375  733.373047  87.899185  3107.084473   
     7 -1661.612183    1027.905151    12.38824   8.876429          NaN   
     8  -470.827209     346.035034    7.571723   5.376757    58.114021   
...             ...            ...         ...        ...          ...   
29 5 5          NaN            NaN         NaN        NaN          NaN   
   6 0          NaN            NaN         NaN        NaN          NaN   
     1          NaN            NaN         NaN        NaN          NaN   
     2          NaN            NaN         NaN        NaN          NaN   
     3          NaN            NaN         NaN        NaN          NaN   

        SIGKFEXTR  R-free-flags      F-model  PHIF-model      2FOFCWT  \
H  K L                                                                  
0  0 2        NaN           NaN          NaN         NaN  6650.478516   
     4        NaN            14          NaN         NaN  1927.483765   
     6  16.496628             6  3358.235596         0.0  3301.962158   
     7        NaN           NaN          NaN         NaN          NaN   
     8  40.554333            17   168.217529         0.0     2.425961   
...           ...           ...          ...         ...          ...   
29 5 5        NaN             3          NaN         NaN          NaN   
   6 0        NaN            10          NaN         NaN          NaN   
     1        NaN             8          NaN         NaN          NaN   
     2        NaN            19          NaN         NaN          NaN   
     3        NaN            13          NaN         NaN          NaN   

        PH2FOFCWT  2FOFCWT_no_fill  PH2FOFCWT_no_fill      FOFCWT  PHFOFCWT  
H  K L                                                                       
0  0 2        0.0              NaN                NaN         NaN       NaN  
     4        0.0              NaN                NaN         NaN       NaN  
     6        0.0      3301.962158                0.0  752.040527       0.0  
     7        NaN              NaN                NaN         NaN       NaN  
     8        0.0         2.425961                0.0  124.879761     180.0  
...           ...              ...                ...         ...       ...  
29 5 5        NaN              NaN                NaN         NaN       NaN  
   6 0        NaN              NaN                NaN         NaN       NaN  
     1        NaN              NaN                NaN         NaN       NaN  
     2        NaN              NaN                NaN         NaN       NaN  
     3        NaN              NaN                NaN         NaN       NaN  

[56146 rows x 15 columns]

In [7]:
map_dark, map_light = get_photolyase_maps(ds_dark, ds_light)



In [15]:
from photolyase import loading_diffmaps
diffmaps, diffmap_config = loading_diffmaps(
    map_dark=map_dark,
    map_light=map_light,
    changing_bit=changing_bit,
    map_sampling=map_sampling,
)


Reading from photolyase30ns.mtz


In [9]:

# key = "tv"
# diffmap = diffmaps[key]

key = "tv"
comparison_key = "vanilla_diffmap"
scale_diffmap = True 


if scale_diffmap:
    diffmap = scale_maps(reference_map=diffmaps[comparison_key], map_to_scale=diffmaps[key])
else:
    diffmap = diffmaps[key]

# pos_blobs2, neg_blobs2 = find_largest_blobs2(diffmap=diffmap, map_sampling=map_sampling,
#                                               threshold=0.5, minimum_size=3)


In [10]:
pos_blobs, neg_blobs = find_largest_blobs(diffmap, map_sampling=map_sampling,
                                          threshold=0.5, minimum_size=3)
pos_blobs2, neg_blobs2 = find_largest_blobs2(diffmap=diffmap, map_sampling=map_sampling,
                                              threshold=0.5, minimum_size=3)

In [16]:
key = "tv"
comparison_key = "vanilla_diffmap"
scale_diffmap = True 


if scale_diffmap:
    diffmap = scale_maps(reference_map=diffmaps[comparison_key], map_to_scale=diffmaps[key])
else:
    diffmap = diffmaps[key]

# pos_blobs2, neg_blobs2 = find_largest_blobs2(diffmap=diffmap, map_sampling=map_sampling,
#                                               threshold=0.5, minimum_size=3)


from systematic_plots import run_plots
rho_shape = diffmap.to_3d_numpy_map(map_sampling=map_sampling).shape
info_container = {
    "tname": "Photolyase",
    "fname": changing_bit,
    "pdbloc_light": pdbloc_light,
    "hs_limit": hs_limit,
    "map_sampling": map_sampling,
    "datatype": "photolyase",
    "mask_loc_sphere": mask_loc_sphere,
    "mask_hand": mask_loc_handcraft,
        # "ball": ccp4_to_mask(mask_loc_sphere, rho_shape),
        # "handpicked": ccp4_to_mask(mask_loc_handcraft, rho_shape),
    "diffmap_config": diffmap_config,
    "fshort": "PL",
    "minimum_threshold": 0.25,
    "mid_xtr_factor": 5,
    "max_xtr_factor": 40,

    "rescaling_diffmaps":scale_diffmap 
    }
# diffmap_path = evaluation_path_basis + f"{key}/"
filestart = f"{info_container['fshort']}_{key}_"

filename_dict = {
            "save_fig": False,
            "display": True,
            "filestart": filestart,
            "rerun": True
            }
function_selection = [
        "many_negsum_best_guess",
        "many_negsum_many_thresh",
        "single_negsum_overview",
        "show_comparison",
    ]


# function_selection = [
#    "many_negsum_best_guess", "many_negsum_many_thresh"]
run_plots(diffmap, map_dark, info_container, key, filename_dict,function_selection=function_selection,)

2025-08-28 09:50:00,538: systematic_plots: INFO: Created extrapolation factors ranging from 1 to 40 with mid at 5 (systematic_plots.py:2071)
2025-08-28 09:50:00,539: systematic_plots: INFO: Setting thresholds from 0.25 to 1 with step 0.05 (systematic_plots.py:2055)
2025-08-28 09:50:00,569: systematic_plots: ERROR: Diffmap 3*sig: 0.04, max negative: -0.44 (systematic_plots.py:2088)


KeyError: 'sigma'

In [14]:
key = "tv"
comparison_key = "vanilla_diffmap"
scale_diffmap = False 


if scale_diffmap:
    diffmap = scale_maps(reference_map=diffmaps[comparison_key], map_to_scale=diffmaps[key])
else:
    diffmap = diffmaps[key]

from systematic_plots import run_plots
rho_shape = diffmap.to_3d_numpy_map(map_sampling=map_sampling).shape
info_container = {
    "tname": "Photolyase",
    "fname": changing_bit,
    "pdbloc_light": pdbloc_light,
    "hs_limit": hs_limit,
    "map_sampling": map_sampling,
    "datatype": "photolyase",
    "mask_loc_sphere": mask_loc_sphere,
    "mask_hand": mask_loc_handcraft,
        # "ball": ccp4_to_mask(mask_loc_sphere, rho_shape),
        # "handpicked": ccp4_to_mask(mask_loc_handcraft, rho_shape),
    "fshort": "PL",
    "minimum_threshold": 0.2,
    "mid_xtr_factor" : 6,
    "max_xtr_factor": 40,
    "sigma" : 4
}

# diffmap_path = evaluation_path_basis + f"{key}/"
filestart = f"{info_container['fshort']}_{key}_"

filename_dict = {
            "save_fig": False,
            "display": True,
            "filestart": filestart,
            "rerun": True
            }
function_selection = [
        "many_negsum_best_guess",
        "many_negsum_many_thresh",
        "single_negsum_overview",
        "show_comparison",
    ]


# function_selection = [
#    "many_negsum_best_guess", "many_negsum_many_thresh"]
run_plots(diffmap, map_dark, info_container, key, filename_dict,function_selection=function_selection,)


2025-08-28 09:49:31,390: systematic_plots: INFO: Created extrapolation factors ranging from 1 to 40 with mid at 6 (systematic_plots.py:2071)
2025-08-28 09:49:31,390: systematic_plots: INFO: Setting thresholds from 0.2 to 1 with step 0.05 (systematic_plots.py:2055)
2025-08-28 09:49:31,421: systematic_plots: ERROR: Diffmap 3*sig: 0.01, max negative: -0.06 (systematic_plots.py:2088)


KeyError: 'diffmap_config'